# 🛡️ Soft Delete Implementation with Metadata

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/07_advanced_patterns/soft_delete_pattern.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/07_advanced_patterns/soft_delete_pattern.ipynb)

## 🏢 Business Scenario

In highly regulated industries like **Finance**, **Healthcare**, or **SaaS**, maintaining a permanent audit trail is a legal requirement. When a user deletes their account or a record is removed from the source system, simply deleting the row in your data lake (a "Hard Delete") is often unacceptable. 

**Stakeholders** (Compliance Officers, Accountants, Data Scientists) need to know:
1. **Who** was in the system at a specific point in time (Historical Truth).
2. **When** exactly a record became inactive.
3. **Why** the record was removed.

## 💡 Value Proposition

*   **Data Integrity**: Prevents accidental loss of historical data that contributes to past financial reports or trend analysis.
*   **Compliance (GDPR/Audit)**: Provides the "Right to be Forgotten" while maintaining a system-level log that the request was processed on a specific date.
*   **Operational Reliability**: Allows data engineers to "undo" deletions easily if a source system error occurred, without needing to restore from a full database backup.
*   **Standardized Analytics**: By using the `_lakelogic_` prefix, every table follows the same audit standard, making it easy for BI tools to filter active records globally.

---

## 🎯 Goals

1. Flag records as deleted using `_lakelogic_is_deleted`.
2. Automatically capture the deletion timestamp in `_lakelogic_deleted_at`.
3. Log the deletion reason in `_lakelogic_delete_reason`.

## 🚀 Step 1: Create the Standardized Contract

We define the standardized system columns in the materialization section.

In [ ]:
contract_yaml = """
version: 1.0.0
dataset: users_silver

source:
  type: table
  cdc_op_field: "op"          
  cdc_delete_values: ["D"]   

primary_key: ["user_id"]

materialization:
  strategy: merge
  path: "./data/users_silver/"
  format: parquet
  soft_delete_column: "_lakelogic_is_deleted"
  soft_delete_value: true
  soft_delete_time_column: "_lakelogic_deleted_at"
  soft_delete_reason_column: "_lakelogic_delete_reason"
"""

with open('soft_delete_contract.yaml', 'w') as f:
    f.write(contract_yaml)

print("✅ Standardized Contract created!")

## ▶️ Step 2: Initial Load

Start with active users.

In [ ]:
from lakelogic.core.processor import DataProcessor
import polars as pl
import os
import shutil

# Clean up previous runs if any
if os.path.exists("./data/users_silver/"):
    shutil.rmtree("./data/users_silver/")

df_v1 = pl.DataFrame([
    {"user_id": 1, "name": "Alice", "op": "I"},
    {"user_id": 2, "name": "Bob", "op": "I"}
])

processor = DataProcessor(contract="soft_delete_contract.yaml", engine="polars")
processor.run(df_v1, materialize=True)

print("\n📂 Silver Table Contents (V1):")
print(pl.read_parquet("./data/users_silver/data.parquet"))

## 🛑 Step 3: Trigger Soft Delete with Metadata

When Bob is deleted, LakeLogic will now automatically populate the `_at` and `_reason` columns.

In [ ]:
df_v2 = pl.DataFrame([
    {"user_id": 2, "name": "Bob", "op": "D"} # Bob is out
])

processor.run(df_v2, materialize=True)

# Final result check
final_df = pl.read_parquet("./data/users_silver/data.parquet")
print("\n📂 Final Silver Table with Metadata:")
print(final_df.select(["user_id", "name", "_lakelogic_is_deleted", "_lakelogic_deleted_at", "_lakelogic_delete_reason"]))
